In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 57.2 MB/s eta 0:00:00


In [ ]:
import gensim.downloader as gen
vw = gen.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
type(vw)

gensim.models.keyedvectors.KeyedVectors

In [ ]:
vw.most_similar('king')

[('kings', 0.7138045430183411),
 ('queen', 0.6510956883430481),
 ('monarch', 0.6413194537162781),
 ('crown_prince', 0.6204220056533813),
 ('prince', 0.6159993410110474),
 ('sultan', 0.5864824056625366),
 ('ruler', 0.5797567367553711),
 ('princes', 0.5646552443504333),
 ('Prince_Paras', 0.5432944297790527),
 ('throne', 0.5422105193138123)]

In [ ]:
vw.most_similar(positive=['king', 'woman'], negative=['man'])

[('queen', 0.7118193507194519),
 ('monarch', 0.6189674139022827),
 ('princess', 0.5902431011199951),
 ('crown_prince', 0.5499460697174072),
 ('prince', 0.5377321839332581),
 ('kings', 0.5236844420433044),
 ('Queen_Consort', 0.5235945582389832),
 ('queens', 0.518113374710083),
 ('sultan', 0.5098593235015869),
 ('monarchy', 0.5087411403656006)]

In [ ]:
vw.most_similar(positive=['france', 'berlin'], negative=['berlin'])

[('spain', 0.6375303268432617),
 ('french', 0.6326055526733398),
 ('germany', 0.6314354538917542),
 ('europe', 0.6264256238937378),
 ('italy', 0.6257959008216858),
 ('england', 0.6120776534080505),
 ('european', 0.6074905395507812),
 ('belgium', 0.5972346067428589),
 ('usa', 0.5948355197906494),
 ('serbia', 0.5805614590644836)]

In [ ]:
# get the vector for king from vw
vw.get_vector('king').shape



(300,)

### Step 1: Prepare Sample Text Data and Labels

For demonstration, let's create a small, artificial dataset of sentences and their categories (e.g., 'positive' or 'negative').

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
import numpy as np
import tensorflow_datasets as tfds
"""
# Sample data
data = {
    'text': [
        'This movie is great and fantastic.',
        'I love this amazing film.',
        'What a wonderful experience!',
        'The acting was terrible.',
        'I hated every boring minute of it.',
        'This is a bad film, very disappointing.'
    ],
    'label': ['positive', 'positive', 'positive', 'negative', 'negative', 'negative']
}
"""
data=tfds.load('imdb_reviews')
##data=data['train']
data = pd.DataFrame(data['train'])



{'train': <_PrefetchDataset element_spec={'label': TensorSpec(shape=(), dtype=tf.int64, name=None), 'text': TensorSpec(shape=(), dtype=tf.string, name=None)}>,
 'test': <_PrefetchDataset element_spec={'label': TensorSpec(shape=(), dtype=tf.int64, name=None), 'text': TensorSpec(shape=(), dtype=tf.string, name=None)}>,
 'unsupervised': <_PrefetchDataset element_spec={'label': TensorSpec(shape=(), dtype=tf.int64, name=None), 'text': TensorSpec(shape=(), dtype=tf.string, name=None)}>}

In [ ]:
df.head(5)

,label,text
0,"tf.Tensor(0, shape=(), dtype=int64)","tf.Tensor(b""This was an absolutely terrible mo..."
1,"tf.Tensor(0, shape=(), dtype=int64)",tf.Tensor(b'I have been known to fall asleep d...
2,"tf.Tensor(0, shape=(), dtype=int64)",tf.Tensor(b'Mann photographs the Alberta Rocky...
3,"tf.Tensor(1, shape=(), dtype=int64)",tf.Tensor(b'This is the kind of film for a sno...
4,"tf.Tensor(1, shape=(), dtype=int64)","tf.Tensor(b'As others have mentioned, all the ..."


### Step 2: Convert Text to Document Vectors

We'll create a function to convert each sentence into a vector by averaging the word vectors of its constituent words. Words not found in `vw` will be ignored.

In [ ]:
def text_to_vec(text, model, vector_size=300):
    words = text.lower().split()
    # Filter out words not in the vocabulary and get their vectors
    word_vectors = [model[word] for word in words if word in model.key_to_index]

    if not word_vectors:
        # Return a zero vector if no words are found in the model
        return np.zeros(vector_size)

    # Average the word vectors to get the document vector
    return np.mean(word_vectors, axis=0)

# Apply the function to create document vectors for each text using df_imdb
df_imdb['text_vector'] = df_imdb['text'].apply(lambda x: text_to_vec(x, vw))

display(df_imdb.head())

,text,label,text_vector
0,This was an absolutely terrible movie. Don't b...,negative,"[0.051473968, 0.04439963, 0.035481278, 0.09683..."
1,"I have been known to fall asleep during films,...",negative,"[0.023448398, 0.026926473, 0.026701337, 0.1154..."
2,Mann photographs the Alberta Rocky Mountains i...,negative,"[0.011017439, 0.02483637, 0.01965705, 0.103437..."
3,This is the kind of film for a snowy Sunday af...,positive,"[0.03520298, 0.022219718, 0.017354643, 0.09415..."
4,"As others have mentioned, all the women that g...",positive,"[0.059885178, 0.03390503, 0.029908922, 0.06904..."


### Step 3: Prepare Data for Machine Learning

Split the dataset into training and testing sets, and separate features (document vectors) from labels.

In [ ]:
# Features (X) are the text vectors, Labels (y) are the categories from df_imdb
X = np.array(df_imdb['text_vector'].tolist())
y = df_imdb['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (4200, 300)
y_train shape: (4200,)
X_test shape: (1800, 300)
y_test shape: (1800,)


### Step 4: Train a Classifier

We'll use a simple Logistic Regression classifier from `scikit-learn`.

In [ ]:
classifier = LogisticRegression(max_iter=1000) # Increased max_iter for convergence
classifier.fit(X_train, y_train)

print("Classifier trained successfully.")

Classifier trained successfully.


### Step 5: Evaluate the Classifier

Predict on the test set and print a classification report to see the performance.

In [ ]:
y_pred = classifier.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

    negative       0.79      0.80      0.80       884
    positive       0.81      0.80      0.80       916

    accuracy                           0.80      1800
   macro avg       0.80      0.80      0.80      1800
weighted avg       0.80      0.80      0.80      1800



In [ ]:
!pip install -q tensorflow_datasets

In [ ]:
import tensorflow_datasets as tfds
import pandas as pd

# Load the IMDB movie review dataset
(ds_train, ds_test), ds_info = tfds.load(
    'imdb_reviews',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

# Convert to pandas DataFrame for easier manipulation
# Take a subset for quicker demonstration, e.g., first 5000 samples
# For full dataset, remove .take(5000)

# For training data
train_texts = [text.numpy().decode('utf-8') for text, _ in ds_train.take(5000)]
train_labels = [label.numpy() for _, label in ds_train.take(5000)]
df_train = pd.DataFrame({'text': train_texts, 'label': train_labels})
df_train['label'] = df_train['label'].map({0: 'negative', 1: 'positive'})

# For testing data (take a smaller subset for demonstration)
test_texts = [text.numpy().decode('utf-8') for text, _ in ds_test.take(1000)]
test_labels = [label.numpy() for _, label in ds_test.take(1000)]
df_test = pd.DataFrame({'text': test_texts, 'label': test_labels})
df_test['label'] = df_test['label'].map({0: 'negative', 1: 'positive'})

df_imdb = pd.concat([df_train, df_test], ignore_index=True)
display(df_imdb.head())
print(f"Loaded {len(df_imdb)} IMDB movie reviews.")
print(f"Value counts for labels:\n{df_imdb['label'].value_counts()}")

,text,label
0,This was an absolutely terrible movie. Don't b...,negative
1,"I have been known to fall asleep during films,...",negative
2,Mann photographs the Alberta Rocky Mountains i...,negative
3,This is the kind of film for a snowy Sunday af...,positive
4,"As others have mentioned, all the women that g...",positive


Loaded 6000 IMDB movie reviews.
Value counts for labels:
label
positive    3023
negative    2977
Name: count, dtype: int64


In [ ]:
# Convert test_texts to document vectors
X_test_vectors = np.array([text_to_vec(text, vw) for text in test_texts])

y_pred = classifier.predict(X_test_vectors)

# Ensure test_labels match the type of y_pred (strings 'negative', 'positive')
# Use the 'label' column from df_test which already has string labels
y_true_str = df_test['label'].tolist()

print("Classification Report:")
print(classification_report(y_true_str, y_pred))

Classification Report:
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67       503
    positive       0.00      0.00      0.00       497

    accuracy                           0.50      1000
   macro avg       0.25      0.50      0.33      1000
weighted avg       0.25      0.50      0.34      1000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
import random

# Select a random review from the test set
random_index = random.randint(0, len(df_test) - 1)
sample_review = df_test.iloc[random_index]['text']
sample_true_label = df_test.iloc[random_index]['label']

print(f"Sample Review (True Label: {sample_true_label}):\n{sample_review}")

# Convert the sample review to a vector
sample_review_vector = text_to_vec(sample_review, vw)

# Reshape for prediction (LogisticRegression expects a 2D array)
sample_review_vector = sample_review_vector.reshape(1, -1)

# Predict the sentiment
predicted_sentiment = classifier.predict(sample_review_vector)[0]

print(f"\nPredicted Sentiment: {predicted_sentiment}")

Sample Review (True Label: positive):
I rented Boogie Nights last week and I could tell you, when I watched the film I had a blast. If you think that when you watch the film you will get sicked by the porn. I mean yes, if your not a porn person who can't bother being by it, than this isn't the film to see. But the thing is, the whole film isn't really about porn. Well halfway through the film is about the porn industry but the other half is about the character development and the bad situations these characters go through. The actors played there roles perfect, especially Mark Wahlberg, John C. Reilly, and William H. Macy. The sex scenes, of course are terrific but mainly focus on the character's hype in porn films until there struggles. Excellent film, one of the best! <br /><br />Hedeen's Outlook: 10/10 **** A+

Predicted Sentiment: negative


In [ ]:
import numpy as np

data = [70, 80, 85, 90, 95]
mean = np.mean(data)
median = np.median(data)
std_dev = np.std(data)
variance = np.var(data)
print(f"Mean: {mean}")
print(f"Median: {median}")
print(f"Standard Deviation: {std_dev}")
print(f"Variance: {variance}")
print(np.var(data, ddof=0))   # Population variance → 74.0
print(np.var(data, ddof=1))
print(np.var(data, ddof=2))
print(np.var(data, ddof=6))
print(np.var(data, ddof=5))

print(((70-84)+(80-84)+(85-84)+(90-84)+(95-84))/5)

Mean: 84.0
Median: 85.0
Standard Deviation: 8.602325267042627
Variance: 74.0
74.0
92.5
123.33333333333333
inf
inf
-0.2


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:4008: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: divide by zero encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### Step-by-step calculation of Population Variance

**Data points (x):** `[70, 80, 85, 90, 95]`

**1. Calculate the Mean (average) of the data:**
   $\text{Mean} (\mu) = \frac{70 + 80 + 85 + 90 + 95}{5} = \frac{420}{5} = 84$

**2. Calculate the difference between each data point and the mean:**
   * $70 - 84 = -14$
   * $80 - 84 = -4$
   * $85 - 84 = 1$
   * $90 - 84 = 6$
   * $95 - 84 = 11$

**3. Square each of these differences:**
   * $(-14)^2 = 196$
   * $(-4)^2 = 16$
   * $1^2 = 1$
   * $6^2 = 36$
   * $11^2 = 121$

**4. Sum the squared differences:**
   $\text{Sum of Squared Differences} = 196 + 16 + 1 + 36 + 121 = 370$

**5. Divide the sum of squared differences by the number of data points (N) to get the Population Variance:**
   $\text{Population Variance} (\sigma^2) = \frac{\text{Sum of Squared Differences}}{N} = \frac{370}{5} = 74$

This is why `np.var(data, ddof=0)` returns `74.0`.

In [ ]:
(70-84)+(80-85)

-2.8

Now that we have a real dataset (`df_imdb`), we can replace the artificial `df` with `df_imdb` in the subsequent steps to convert text to vectors and train the classifier. I will modify the existing `Step 2` to use this new dataframe.

This example uses a very small dataset for illustration. In a real-world scenario, you would use a much larger, more diverse dataset, perform more extensive text preprocessing (like stemming, lemmatization, stop word removal), and potentially use more sophisticated methods for creating document embeddings (e.g., TF-IDF weighted averaging, Doc2Vec, or more complex neural network architectures like LSTMs or Transformers if you were training a deep learning model).